# Week 8: K Nearest Neighbors
**Rupal Paliwal, Mod C, Semester 1**

**Data:** Part 1 uses the box office data (4,803 films). Part 2 adds MovieLens and the streaming catalogs (2,407 films from 2000 to 2015) and the MovieLens tag genome.

**Question:** Can a film's revenue be predicted from the films most similar to it? Does the way we measure "similar" (the distance) matter?

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import accuracy_score, r2_score

# Part 1: Box Office Data

## 1. Load the data

Same features as earlier weeks: log vote count, rating, year, genres, and language. Two targets: **hit** (worldwide gross above the median) and **log revenue**.

In [2]:
df = pd.read_csv('enhanced_box_office_data_2000-2024_u.csv')
data = df.copy()
data['Rating_num'] = data['Rating'].str.split('/').str[0].astype(float)
data = data.dropna(subset=['Rating_num', 'Vote_Count', 'Genres', 'Original_Language'])
data = data[data['Rating_num'] > 0]

X = pd.DataFrame({'log_votes': np.log1p(data['Vote_Count']), 'rating': data['Rating_num'], 'year': data['Year']}, index=data.index)
genres = data['Genres'].str.get_dummies(sep=', ').add_prefix('genre_')
top_langs = data['Original_Language'].value_counts().index[:10]
lang = data['Original_Language'].where(data['Original_Language'].isin(top_langs), 'other')
langs = pd.get_dummies(lang, prefix='lang').drop(columns='lang_en').astype(int)
X = pd.concat([X, genres, langs], axis=1)
y_hit = (data['$Worldwide'] > df['$Worldwide'].median()).astype(int)
y_rev = np.log(data['$Worldwide'])
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X, y_hit, test_size=0.2, random_state=42, stratify=y_hit)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X, y_rev, test_size=0.2, random_state=42)
skf = StratifiedKFold(5, shuffle=True, random_state=42)
kf = KFold(5, shuffle=True, random_state=42)
print('Films:', len(X), '| features:', X.shape[1])

Films: 4803 | features: 32


## 2. Feature scaling

KNN finds neighbors by distance, so a feature with big numbers (year, around 2,000 to 2,024) would swamp features measured from 0 to 1 (the genre and language columns). Standardizing gives every feature the same weight. I compare three versions with k = 15, scored by 5 fold cross validation (CV).

In [3]:
no_genre = [c for c in X.columns if not c.startswith('genre_')]     # votes, rating, year, and the language columns
versions = [('No scaling, all 32 features', KNeighborsClassifier(n_neighbors=15), list(X.columns)),
            ('Standardized, all 32 features', make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)), list(X.columns)),
            ('Standardized, no genre columns (13)', make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)), no_genre)]
for name, model, cols in versions:
    cv = cross_val_score(model, Xc_train[cols], yc_train, cv=skf).mean()
    print(f'{name:37} CV accuracy: {cv:.4f}')

No scaling, all 32 features           CV accuracy: 0.7681
Standardized, all 32 features         CV accuracy: 0.7277


Standardized, no genre columns (13)   CV accuracy: 0.7801


Standardizing all 32 features made KNN *worse*. Once every column counts equally, the 19 genre columns (most of which barely relate to revenue) make up most of the distance, so a film's "nearest neighbors" are mostly films with the same genres, not films with similar reach. Without scaling, year and vote count dominated the distance by accident, which happened to help. The best version standardizes but drops the genre columns. KNN has no way to learn which features matter, so choosing the features is part of the model. I use the 13 feature version from here on.

## 3. Choosing k

A small k copies the closest few films and memorizes noise. A large k averages over many films and blurs real differences. I pick k with 5 fold cross validation (CV).

In [4]:
rows = []
for k in [1, 3, 5, 11, 21, 41, 81, 161]:
    m = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    cv = cross_val_score(m, Xc_train[no_genre], yc_train, cv=skf).mean()
    train = accuracy_score(yc_train, m.fit(Xc_train[no_genre], yc_train).predict(Xc_train[no_genre]))
    rows.append({'k': k, 'train_accuracy': round(train, 4), 'CV_accuracy': round(cv, 4)})
k_table = pd.DataFrame(rows)
k_table

,k,train_accuracy,CV_accuracy
0,1,0.9992,0.7353
1,3,0.8696,0.7678
2,5,0.8417,0.7788
3,11,0.8186,0.7842
4,21,0.8061,0.7808
5,41,0.7936,0.7782
6,81,0.7866,0.7790
7,161,0.7788,0.7743


## 4. Choosing the distance

- **Euclidean** (Minkowski p = 2): squares each difference, so one big difference counts a lot.
- **Manhattan** (Minkowski p = 1): adds up absolute differences, so one big difference counts less.
- **Minkowski** with other p values.
- **Cosine**: only the direction of the feature vector matters, not its size.

I tune k and the distance together with CV, for both targets.

In [5]:
grid = [{'kneighborsclassifier__n_neighbors': [11, 21, 41, 81], 'kneighborsclassifier__p': [1, 1.5, 2, 3]},
        {'kneighborsclassifier__n_neighbors': [11, 21, 41, 81], 'kneighborsclassifier__metric': ['cosine'],
         'kneighborsclassifier__algorithm': ['brute']}]
clf_search = GridSearchCV(make_pipeline(StandardScaler(), KNeighborsClassifier()), grid, cv=skf, n_jobs=-1).fit(Xc_train[no_genre], yc_train)
res = pd.DataFrame(clf_search.cv_results_)
res['distance'] = res['param_kneighborsclassifier__metric'].fillna('minkowski p=' + res['param_kneighborsclassifier__p'].astype(str))
print('Best CV accuracy for each distance:')
print(res.groupby('distance')['mean_test_score'].max().round(4).to_string())
print('\nBest settings:', clf_search.best_params_)
print('Test accuracy:', round(accuracy_score(yc_test, clf_search.predict(Xc_test[no_genre])), 4))

Best CV accuracy for each distance:
distance
cosine             0.7824
minkowski p=1.0    0.7821
minkowski p=1.5    0.7816
minkowski p=2.0    0.7842
minkowski p=3.0    0.7866

Best settings: {'kneighborsclassifier__n_neighbors': 11, 'kneighborsclassifier__p': 3}


Test accuracy: 0.7638


In [6]:
reg_grid = {'kneighborsregressor__n_neighbors': [5, 11, 21, 41], 'kneighborsregressor__p': [1, 2],
            'kneighborsregressor__weights': ['uniform', 'distance']}
reg_search = GridSearchCV(make_pipeline(StandardScaler(), KNeighborsRegressor()), reg_grid, cv=kf, scoring='r2', n_jobs=-1).fit(Xr_train[no_genre], yr_train)
print('Best settings:', reg_search.best_params_)
print('CV R2:  ', round(reg_search.best_score_, 4))
print('Test R2:', round(r2_score(yr_test, reg_search.predict(Xr_test[no_genre])), 4))

Best settings: {'kneighborsregressor__n_neighbors': 11, 'kneighborsregressor__p': 1, 'kneighborsregressor__weights': 'uniform'}
CV R2:   0.6323
Test R2: 0.5971


## 5. How KNN compares (same test films)

In [7]:
pd.DataFrame({'model': ['Logistic regression (Week 4)', 'Random forest (Week 6)', 'Gradient boosting (Week 9)', 'KNN, tuned'],
              'hit_accuracy': [0.7825, 0.8012, 0.7877, round(accuracy_score(yc_test, clf_search.predict(Xc_test[no_genre])), 4)],
              'revenue_R2': [0.5256, 0.6839, 0.7011, round(r2_score(yr_test, reg_search.predict(Xr_test[no_genre])), 4)]})

,model,hit_accuracy,revenue_R2
0,Logistic regression (Week 4),0.7825,0.5256
1,Random forest (Week 6),0.8012,0.6839
2,Gradient boosting (Week 9),0.7877,0.7011
3,"KNN, tuned",0.7638,0.5971


# Part 2: All Three Datasets

The 2,407 films from 2000 to 2015 that matched MovieLens, using the tuned settings from Part 1. As in Part 1, the genre columns are left out: box office only has 13 features, and all three datasets add 7 (MovieLens rating count, average rating, and tag count, and a flag for each streaming service).

In [8]:
combo = pd.read_csv('combined_films.csv')          # built by OMDS-ModC-DataPrep-Paliwal-Rupal.ipynb
sub = combo.loc[X.index]
sub = sub[(sub['in_movielens'] == 1) & (sub['ml_n_ratings'] > 0)]
extra = pd.DataFrame({
    'ml_log_ratings': np.log1p(sub['ml_n_ratings']), 'ml_mean_rating': sub['ml_mean_rating'],
    'ml_log_tags': np.log1p(sub['ml_n_tags']),
    'on_netflix': sub['on_netflix'], 'on_prime': sub['on_prime'], 'on_hulu': sub['on_hulu'], 'on_disney': sub['on_disney'],
}, index=sub.index)
XA = X.loc[sub.index]                  # box office features only
XB = pd.concat([XA, extra], axis=1)    # all three datasets
print('Films:', len(sub), '| box office features:', XA.shape[1], '| all three datasets:', XB.shape[1])
y3_hit = (sub['$Worldwide'] > df['$Worldwide'].median()).astype(int)
y3_rev = np.log(sub['$Worldwide'])
trc, tec = train_test_split(XA.index, test_size=0.2, random_state=42, stratify=y3_hit)
trr, ter = train_test_split(XA.index, test_size=0.2, random_state=42)
rows = []
ml_cols = ['ml_log_ratings', 'ml_mean_rating', 'ml_log_tags']
stream_cols = ['on_netflix', 'on_prime', 'on_hulu', 'on_disney']
for name, Z in [('Box office only', XA[no_genre]),
                ('+ MovieLens (3 features)', XB[no_genre + ml_cols]),
                ('+ MovieLens + streaming flags (7 features)', XB[no_genre + ml_cols + stream_cols])]:
    c = clf_search.best_estimator_.fit(Z.loc[trc], y3_hit[trc])
    acc = accuracy_score(y3_hit[tec], c.predict(Z.loc[tec]))
    r = reg_search.best_estimator_.fit(Z.loc[trr], y3_rev[trr])
    rows.append({'features': name, 'hit_accuracy': round(acc, 4), 'revenue_R2': round(r2_score(y3_rev[ter], r.predict(Z.loc[ter])), 4)})
pd.DataFrame(rows)

Films: 2407 | box office features: 32 | all three datasets: 39


,features,hit_accuracy,revenue_R2
0,Box office only,0.8154,0.6543
1,+ MovieLens (3 features),0.8008,0.6531
2,+ MovieLens + streaming flags (7 features),0.7905,0.6431


## P2.2 Similar films by content (cosine distance on the tag genome)

The tag genome scores each film on 1,128 tags. Cosine distance compares the *pattern* of tags, which suits this data: two films can have the same mix of tags even if one scores higher on everything. First, a sanity check: the 5 films most similar to one well known film.

In [9]:
genome = pd.read_csv('genome_features.csv', index_col=0)
gsub = sub[sub['ml_movieId'].astype(int).isin(genome.index)]
G = genome.loc[gsub['ml_movieId'].astype(int)].set_index(gsub.index)
titles = gsub['Release Group']

from sklearn.neighbors import NearestNeighbors
nn = NearestNeighbors(n_neighbors=6, metric='cosine').fit(G)
for film in ['Toy Story 3', 'The Dark Knight']:
    i = titles[titles == film].index[0]
    dist, idx = nn.kneighbors(G.loc[[i]])
    print(film, '->', list(titles.iloc[idx[0][1:]]))

Toy Story 3 -> ['Up', 'Monsters, Inc.', 'Finding Nemo', 'Ratatouille', 'How to Train Your Dragon']
The Dark Knight -> ['Batman Begins', 'The Dark Knight Rises', 'Watchmen', 'Sin City', 'V for Vendetta']


In [10]:
y_g = y3_rev[gsub.index]
trg, teg = train_test_split(G.index, test_size=0.2, random_state=42)
rows = []
for metric in ['euclidean', 'manhattan', 'cosine']:
    for k in [5, 10, 20, 40]:
        m = KNeighborsRegressor(n_neighbors=k, metric=metric, algorithm='brute', weights='distance')
        cv = cross_val_score(m, G.loc[trg], y_g[trg], cv=kf, scoring='r2').mean()
        rows.append({'distance': metric, 'k': k, 'CV_R2': round(cv, 4)})
g_table = pd.DataFrame(rows)
best = g_table.loc[g_table['CV_R2'].idxmax()]
final = KNeighborsRegressor(n_neighbors=int(best['k']), metric=best['distance'], algorithm='brute', weights='distance').fit(G.loc[trg], y_g[trg])
print('Best:', best['distance'], 'k =', int(best['k']), '| test R2 (genome tags only):', round(r2_score(y_g[teg], final.predict(G.loc[teg])), 4))
g_table.pivot(index='k', columns='distance', values='CV_R2')

Best: manhattan k = 10 | test R2 (genome tags only): 0.4649


distance,cosine,euclidean,manhattan
k,,,
5,0.4398,0.4583,0.4995
10,0.4623,0.4929,0.5137
20,0.4693,0.4744,0.5007
40,0.4528,0.4400,0.4631


## Think About It: two quick checks in Python

In [11]:
# What does the Minkowski distance do when p is huge?
def minkowski_2d(x, y, p):
    big, small = max(x, y), min(x, y)
    return big * (1 + (small / big) ** p) ** (1 / p)   # equals (x**p + y**p)**(1/p), written so huge powers don't overflow

for x, y in [(3, 4), (1, 10), (5, 5)]:
    print(f'x={x}, y={y}:', {p: round(minkowski_2d(x, y, p), 4) for p in [1, 2, 10, 100, 1_000_000]})

x=3, y=4: {1: 7.0, 2: 5.0, 10: 4.022, 100: 4.0, 1000000: 4.0}
x=1, y=10: {1: 11.0, 2: 10.0499, 10: 10.0, 100: 10.0, 1000000: 10.0}
x=5, y=5: {1: 10.0, 2: 7.0711, 10: 5.3589, 100: 5.0348, 1000000: 5.0}


In [12]:
# Standardizing does not remove outliers: z scores keep the same shape, so extreme films stay extreme
votes_z = (data['Vote_Count'] - data['Vote_Count'].mean()) / data['Vote_Count'].std()
print('Largest z score for vote count:', round(votes_z.max(), 1), '| films more than 3 SD above the mean:', int((votes_z > 3).sum()))

Largest z score for vote count: 8.7 | films more than 3 SD above the mean: 123


# Conclusions

**KNN is simple but not the strongest model here.** Tuned, it reached 76.4% test accuracy for hits (CV 78.7%) and a test R² of 0.597 for revenue. That beats linear regression (0.526) but trails the random forest (80.1%, 0.684) and gradient boosting (0.701).

**Choosing features mattered more than choosing the distance.** Standardizing all 32 features *lowered* CV accuracy from 76.8% to 72.8%, because the 19 genre columns then made up most of the distance. Dropping them raised it to 78.0%. The distance choice barely mattered: Euclidean, Manhattan, Minkowski p = 3, and cosine all scored within half a point of each other in CV.

**k controls overfitting.** With k = 1, KNN scored 99.9% on the training films (each film is its own nearest neighbor) but only 73.5% in CV. CV accuracy peaked around k = 11 and slowly fell for larger k as the neighborhoods became too broad.

**Part 2 (all three datasets).** Unlike every model in earlier weeks, KNN got slightly *worse* when MovieLens and streaming features were added (81.5% to 79.1% accuracy, R² 0.654 to 0.643). KNN can't learn that a feature is weak, so each extra column adds noise to every distance. On the tag genome, the nearest neighbors make sense ("Toy Story 3" is closest to "Up", "Monsters, Inc.", and "Finding Nemo"), but predicting revenue from the 10 most similar films only reached a test R² of 0.465, far below gradient boosting (0.822). Films with similar content can earn very different amounts.

### Think About It

- **Why use Manhattan distance when there are outliers?** Euclidean squares each difference, so one feature where two films are very far apart can dominate the whole distance. Manhattan adds the differences without squaring, so one extreme feature counts less.
- **What does Minkowski distance do when p is very large?** It becomes the single largest difference (the "Chebyshev" or p = infinity distance). The Python check above shows (3^p + 4^p)^(1/p) going 7, 5, 4.02, 4.0, 4.0 as p grows: only the bigger number survives.
- **Can there be outliers after standardizing?** Yes. Standardizing only shifts and rescales each feature, so every film keeps the same position relative to the others. The check above shows the most voted film has a z score of 8.7, and 123 films are more than 3 standard deviations above the mean. Standardizing changes the units, not the shape.
- **Why use cosine distance for documents?** Cosine compares the direction of two vectors and ignores their length. A long document and a short one about the same topic have word counts in the same proportions, so they point the same way and get a cosine distance near 0, while Euclidean or Manhattan distance would call them far apart just because one is longer.